In [8]:
# ============================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

# Path to your .env file
env_path = Path(r"F:\AI_ATS_SCANNER\.env")

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        f"GOOGLE_API_KEY not found in {env_path}. "
        "Please check your .env file."
    )

os.environ["GOOGLE_API_KEY"] = api_key

print("GOOGLE_API_KEY loaded successfully.")

GOOGLE_API_KEY loaded successfully.


In [9]:
# ============================================================
# 2. DEFINE FILE PATHS AND CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd
import json
import time
import logging
import os

base_folder = Path(r"F:\AI_ATS_SCANNER")

jd_path = base_folder / "data" / "JD.txt"

# CSV produced by 02_filtering.ipynb
input_csv = (
    base_folder
    / "data"
    / "resumes"
    / "analysis_results"
    / "shortlisted_candidates.csv"
)

# Markdown files produced by 01_data_extraction.ipynb
resume_text_folder = (
    base_folder
    / "data"
    / "resumes"
    / "extracted_resumes"
)

# Log scoring errors separately
error_log = (
    base_folder
    / "data"
    / "resumes"
    / "analysis_results"
    / "scoring_errors.log"
)

REQUEST_DELAY_SECONDS = 1

if not jd_path.exists():
    raise FileNotFoundError(f"Job Description not found: {jd_path}")

if not input_csv.exists():
    raise FileNotFoundError(
        f"Filtered candidates CSV not found: {input_csv}\n"
        "Run 02_filtering.ipynb first."
    )

if not resume_text_folder.exists():
    raise FileNotFoundError(
        f"Extracted resume folder not found: {resume_text_folder}\n"
        "Run 01_data_extraction.ipynb first."
    )

logging.basicConfig(
    filename=str(error_log),
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8"
)

print("Paths validated successfully.")
print(f"Input CSV: {input_csv}")
print(f"Resume text folder: {resume_text_folder}")

Paths validated successfully.
Input CSV: F:\AI_ATS_SCANNER\data\resumes\analysis_results\shortlisted_candidates.csv
Resume text folder: F:\AI_ATS_SCANNER\data\resumes\extracted_resumes


In [10]:
# ============================================================
# 3. READ THE JOB DESCRIPTION AND FILTERED CANDIDATES
# ============================================================

jd_text = jd_path.read_text(encoding="utf-8").strip()

if not jd_text:
    raise ValueError("Job Description file is empty.")

df = pd.read_csv(input_csv, encoding="utf-8-sig")

if df.empty:
    raise ValueError("The filtered candidates CSV has no candidates.")

if "file_name" not in df.columns:
    raise ValueError(
        "The filtered CSV must contain a 'file_name' column "
        "to locate each extracted resume."
    )

# Ensure every candidate has a unique candidate_id.
if "candidate_id" not in df.columns:
    df.insert(
        0,
        "candidate_id",
        [f"CAND_{i:04d}" for i in range(1, len(df) + 1)]
    )
else:
    missing_ids = df["candidate_id"].isna() | (
        df["candidate_id"].astype(str).str.strip() == ""
    )

    for row_index in df.index[missing_ids]:
        df.at[row_index, "candidate_id"] = (
            f"CAND_{row_index + 1:04d}"
        )

if df["candidate_id"].astype(str).duplicated().any():
    raise ValueError(
        "candidate_id values must be unique in the filtered CSV."
    )

# Add only the three component scores and final score.
# Preserve all existing candidate information.
scoring_columns = [
    "resume_quality_score",
    "skills_projects_experience_score",
    "jd_match_score",
    "final_score"
]

for column in scoring_columns:
    if column not in df.columns:
        df[column] = pd.NA

# Save the updated CSV before scoring begins.
df.to_csv(input_csv, index=False, encoding="utf-8-sig")

print(f"Candidates loaded: {len(df)}")
print(f"Job Description characters: {len(jd_text)}")

display(df.head())

Candidates loaded: 3
Job Description characters: 2888


,candidate_id,file_name,experience_years,experience_months,python,sql,machine_learning,total_experience_years,skills_matched,resume_quality_score,skills_projects_experience_score,jd_match_score,final_score
0,CAND_0001,Aditi_Sarang_Resume.md,0,0,1,1,1,0.000000,3,<NA>,<NA>,<NA>,<NA>
1,CAND_0002,Sanika_Deshmukh_Resume.md,0,0,1,1,1,0.000000,3,<NA>,<NA>,<NA>,<NA>
2,CAND_0003,Harshit_Jhalani_Resume.md,1,7,1,1,1,1.583333,3,<NA>,<NA>,<NA>,<NA>


In [11]:
# ============================================================
# 4. MAP EXTRACTED RESUME FILES TO CANDIDATES
# ============================================================

# The extracted resume text is stored as Markdown (.md) files.
# Match each candidate's file_name to its corresponding Markdown file.

markdown_files = sorted(resume_text_folder.rglob("*.md"))

markdown_by_stem = {}

for md_path in markdown_files:
    markdown_by_stem.setdefault(
        md_path.stem.casefold(), []
    ).append(md_path)


def find_resume_text_path(file_name):
    """
    Find the extracted Markdown file for a candidate's resume.
    """

    stem = Path(str(file_name)).stem.casefold()
    matches = markdown_by_stem.get(stem, [])

    if len(matches) == 0:
        raise FileNotFoundError(
            f"No extracted Markdown text found for: {file_name}"
        )

    if len(matches) > 1:
        raise ValueError(
            f"Multiple Markdown files found for {file_name}: "
            f"{[str(path) for path in matches]}"
        )

    return matches[0]


print(f"Extracted Markdown files found: {len(markdown_files)}")

Extracted Markdown files found: 6


In [12]:
# ============================================================
# 5. INITIALIZE GEMINI AND STRUCTURED OUTPUT PARSER
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

# Initialize JSON output parser
parser = JsonOutputParser()

print("Gemini initialized successfully.")

Gemini initialized successfully.


In [13]:
# ============================================================
# 6. DEFINE THE SCORING PROMPT
# ============================================================

scoring_prompt = PromptTemplate(
    template="""
You are an AI resume evaluator. Evaluate the candidate against the
provided Job Description (JD).

JOB DESCRIPTION:
{jd_text}

CANDIDATE ID:
{candidate_id}

RESUME TEXT:
{resume_text}

Evaluate the candidate on the following three criteria.

1. OVERALL RESUME QUALITY (20% weight)
Score: 0-100
Evaluate clarity, completeness, organization, specificity of achievements,
and how effectively the resume communicates the candidate's background.
Do not reward fancy formatting, visual design, or resume length alone.

2. SKILLS, PROJECTS & EXPERIENCE (40% weight)
Score: 0-100
Evaluate demonstrated technical skills, technical depth, project complexity,
practical implementation, individual contribution, relevant work experience,
and demonstrated outcomes.
Assess the candidate's overall technical capability, not just JD fit.
Do not treat a skill's name alone as proof of proficiency.

3. JD MATCHING SCORE (40% weight)
Score: 0-100
Evaluate how closely the candidate's demonstrated skills, projects,
experience, qualifications, and responsibilities match the JD.
Consider required and preferred skills and relevant responsibilities.

SCORING RULES:
- Apply the same standards to every candidate.
- Use only information supported by the resume.
- Do not invent skills, experience, achievements, or qualifications.
- Treat missing information as unknown, not automatically as proof
  that the candidate lacks a capability.
- Do not consider protected or sensitive personal characteristics.
- Return integer scores between 0 and 100.
- Do not calculate the final weighted score. Python will calculate it.
- Return only valid JSON. Do not include Markdown or additional text.

Return exactly this JSON structure:
{{
    "candidate_id": "{candidate_id}",
    "resume_quality_score": 0,
    "skills_projects_experience_score": 0,
    "jd_match_score": 0
}}
""",
    input_variables=["jd_text", "candidate_id", "resume_text"]
)

# Create the scoring chain
chain = scoring_prompt | llm | parser

print("Scoring prompt and chain are ready.")

Scoring prompt and chain are ready.


In [17]:
# ============================================================
# 7. VALIDATE SCORES AND CALCULATE FINAL SCORE
# ============================================================

def validate_scoring_response(response, expected_candidate_id):
    """
    Validate the LLM response before saving the scores.
    """

    required_keys = [
        "candidate_id",
        "resume_quality_score",
        "skills_projects_experience_score",
        "jd_match_score"
    ]

    if not isinstance(response, dict):
        raise ValueError("LLM response must be a JSON object.")

    missing_keys = set(required_keys) - set(response.keys())

    if missing_keys:
        raise ValueError(
            f"Missing fields in LLM response: {missing_keys}"
        )

    if str(response["candidate_id"]) != str(expected_candidate_id):
        raise ValueError("Candidate ID mismatch in LLM response.")

    score_keys = [
        "resume_quality_score",
        "skills_projects_experience_score",
        "jd_match_score"
    ]

    for key in score_keys:
        value = response[key]

        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError(f"{key} must be numeric.")

        if not 0 <= value <= 100:
            raise ValueError(f"{key} must be between 0 and 100.")

        if not float(value).is_integer():
            raise ValueError(f"{key} must be an integer.")

    return response


def calculate_final_score(
    resume_quality_score,
    skills_projects_experience_score,
    jd_match_score
):
    """
    Calculate the weighted final score using Python.
    """

    final_score = (
        resume_quality_score * 0.20
        + skills_projects_experience_score * 0.40
        + jd_match_score * 0.40
    )

    return round(final_score, 2)


# Test the final score calculation
test_score = calculate_final_score(79, 89, 69)

print(f"Test final score: {test_score}/100")


Test final score: 79.0/100


In [18]:
# ============================================================
# 8. SCORE CANDIDATES AND SAVE PROGRESS
# ============================================================

# Process candidates one by one.
# Save the CSV after every candidate so completed scores are retained.

import time
import os
import logging

# Log errors separately without adding extra columns to the CSV.
logging.basicConfig(
    filename=str(error_log),
    level=logging.ERROR,
    format="%(asctime)s | %(levelname)s | %(message)s",
    encoding="utf-8"
)

# Skip candidates that already have all four scores.
completed_mask = (
    df["resume_quality_score"].notna()
    & df["skills_projects_experience_score"].notna()
    & df["jd_match_score"].notna()
    & df["final_score"].notna()
)

pending_indices = df.index[~completed_mask].tolist()

print(f"Total candidates: {len(df)}")
print(f"Already scored: {completed_mask.sum()}")
print(f"Pending candidates: {len(pending_indices)}")

for process_number, row_index in enumerate(pending_indices, start=1):

    candidate_id = str(df.at[row_index, "candidate_id"])
    file_name = str(df.at[row_index, "file_name"])

    print(
        f"\n[{process_number}/{len(pending_indices)}] "
        f"Scoring {candidate_id}: {file_name}"
    )

    try:
        # ----------------------------------------------------
        # A. Fetch the extracted resume text
        # ----------------------------------------------------
        resume_text_path = find_resume_text_path(file_name)

        resume_text = resume_text_path.read_text(
            encoding="utf-8"
        ).strip()

        if not resume_text:
            raise ValueError("Extracted resume text is empty.")

        # ----------------------------------------------------
        # B. Send JD and resume text to Gemini
        # ----------------------------------------------------
        response = chain.invoke({
            "jd_text": jd_text,
            "candidate_id": candidate_id,
            "resume_text": resume_text
        })

        # ----------------------------------------------------
        # C. Validate the response
        # ----------------------------------------------------
        response = validate_scoring_response(
            response,
            expected_candidate_id=candidate_id
        )

        # ----------------------------------------------------
        # D. Calculate final score using Python
        # ----------------------------------------------------
        final_score = calculate_final_score(
            response["resume_quality_score"],
            response["skills_projects_experience_score"],
            response["jd_match_score"]
        )

        # ----------------------------------------------------
        # E. Update the correct candidate row
        # ----------------------------------------------------
        df.at[row_index, "resume_quality_score"] = (
            response["resume_quality_score"]
        )

        df.at[row_index, "skills_projects_experience_score"] = (
            response["skills_projects_experience_score"]
        )

        df.at[row_index, "jd_match_score"] = (
            response["jd_match_score"]
        )

        df.at[row_index, "final_score"] = final_score

        # ----------------------------------------------------
        # F. Save progress immediately
        # ----------------------------------------------------
        temp_csv = input_csv.with_suffix(".tmp.csv")

        df.to_csv(
            temp_csv,
            index=False,
            encoding="utf-8-sig"
        )

        os.replace(temp_csv, input_csv)

        print(f"SUCCESS | Final score: {final_score}/100")

    except Exception as e:
        # Record the failure in the log, without adding CSV columns.
        logging.exception(
            "Scoring failed for candidate_id=%s, file_name=%s",
            candidate_id,
            file_name
        )

        # Save the current CSV to preserve earlier successful scores.
        df.to_csv(
            input_csv,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"FAILED | {candidate_id} | {e}")
        print("Error logged. Continuing with the next candidate.")

    time.sleep(1)

print("\nScoring process completed.")

Total candidates: 3
Already scored: 0
Pending candidates: 3

[1/3] Scoring CAND_0001: Aditi_Sarang_Resume.md
SUCCESS | Final score: 79.2/100

[2/3] Scoring CAND_0002: Sanika_Deshmukh_Resume.md
SUCCESS | Final score: 86.2/100

[3/3] Scoring CAND_0003: Harshit_Jhalani_Resume.md
SUCCESS | Final score: 78.8/100

Scoring process completed.


In [20]:
# ============================================================
# 9. DISPLAY SCORING SUMMARY
# ============================================================

# Reload the latest results from the CSV.
final_df = pd.read_csv(input_csv, encoding="utf-8-sig")

# Convert scoring columns to numeric values for reporting.
score_columns = [
    "resume_quality_score",
    "skills_projects_experience_score",
    "jd_match_score",
    "final_score"
]

for column in score_columns:
    final_df[column] = pd.to_numeric(
        final_df[column],
        errors="coerce"
    )

# Identify successfully scored and pending candidates.
completed_mask = final_df[score_columns].notna().all(axis=1)

print("=" * 60)
print("SCORING SUMMARY")
print("=" * 60)

print(f"Total candidates       : {len(final_df)}")
print(f"Successfully scored    : {completed_mask.sum()}")
print(f"Pending/failed scoring : {(~completed_mask).sum()}")
print(f"Updated CSV            : {input_csv}")
print(f"Error log              : {error_log}")

# Display the four scores alongside candidate identifiers.
display(
    final_df
)

SCORING SUMMARY
Total candidates       : 3
Successfully scored    : 3
Pending/failed scoring : 0
Updated CSV            : F:\AI_ATS_SCANNER\data\resumes\analysis_results\shortlisted_candidates.csv
Error log              : F:\AI_ATS_SCANNER\data\resumes\analysis_results\scoring_errors.log


,candidate_id,file_name,experience_years,experience_months,python,sql,machine_learning,total_experience_years,skills_matched,resume_quality_score,skills_projects_experience_score,jd_match_score,final_score
0,CAND_0001,Aditi_Sarang_Resume.md,0,0,1,1,1,0.000000,3,78,75,84,79.2
1,CAND_0002,Sanika_Deshmukh_Resume.md,0,0,1,1,1,0.000000,3,83,84,90,86.2
2,CAND_0003,Harshit_Jhalani_Resume.md,1,7,1,1,1,1.583333,3,82,76,80,78.8


In [22]:
# ============================================================
# 10. SORT CANDIDATES BY FINAL SCORE AND SAVE
# ============================================================

# Sort candidates from highest to lowest final score.
# Candidates without a final score will appear at the bottom.

df["final_score"] = pd.to_numeric(
    df["final_score"],
    errors="coerce"
)

df = df.sort_values(
    by="final_score",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

# Save the sorted results to the same CSV.
df.to_csv(
    input_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Candidates sorted and saved successfully.")
print(f"Output file: {input_csv}")

# Display the ranked candidates.
display(
    df
)

Candidates sorted and saved successfully.
Output file: F:\AI_ATS_SCANNER\data\resumes\analysis_results\shortlisted_candidates.csv


,candidate_id,file_name,experience_years,experience_months,python,sql,machine_learning,total_experience_years,skills_matched,resume_quality_score,skills_projects_experience_score,jd_match_score,final_score
0,CAND_0002,Sanika_Deshmukh_Resume.md,0,0,1,1,1,0.000000,3,83,84,90,86.2
1,CAND_0001,Aditi_Sarang_Resume.md,0,0,1,1,1,0.000000,3,78,75,84,79.2
2,CAND_0003,Harshit_Jhalani_Resume.md,1,7,1,1,1,1.583333,3,82,76,80,78.8
